# Credit Card Fraud Detection Proof of Concept
Financial institutions process millions of transactions each day, making it difficult to manually identify suspicious activity. Machine learning provides a way to analyze large volumes of transactions and highlight patterns that may indicate fraud.

This proof of concept demonstrates how Microsoft Azure Machine Learning can be used to support fraud monitoring.

The workflow presented in this analysis:
1. Uses historical transaction data as the foundation for analysis
2. Prepares and organizes the data for fraud detection
3. Applies machine learning to identify unusual transaction patterns
4. Evaluates how effectively suspicious activity can be flagged

The goal is to illustrate how machine learning can support fraud monitoring by helping financial institutions identify higher-risk transactions that may require further investigation.

## Fraud Detection Process Overview
The diagram below shows the overall workflow used in this proof-of-concept. It illustrates how transaction data moves through each stage—from environment setup and data preparation to fraud detection, evaluation, and visualization of results.

![Fraud Detection Process](img/pipeline.png)

## Azure Machine Learning Components Used
The table below summarizes the key Azure Machine Learning components involved in this proof-of-concept and the role each one plays in the fraud detection process.

| Azure ML Component | What It Does in This Process                                               |
| ------------------ | -------------------------------------------------------------------------- |
| Azure ML Workspace | The central cloud environment where the fraud detection project is managed |
| Dataset            | Stores the credit card transaction data used for analysis                  |
| Registered Model   | Saves the trained fraud detection process so it can be reused later        |


---
## Workflow

### Step 1: Set Up the Working Environment
The fraud detection process begins by preparing the environment where the analysis will run.
- Connects to the **Azure Machine Learning workspace**
- Loads the Python tools required for data analysis and machine learning

**Why this step matters:**
A stable and centralized environment ensures the analysis runs consistently and all project resources are managed in one secure location.

In [1]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

---
### Step 2: Load the Transaction Data

Next, the credit card transaction dataset is retrieved for analysis.
- Connects to the Azure Machine Learning workspace
- Loads the transaction dataset into the analysis environment

**Business value:**
Working from a controlled dataset ensures the fraud review process starts with reliable and consistent information.

#### Part 1: Environment Verification

The current working directory and available files are checked to confirm that the required configuration file is present before accessing the dataset.

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


#### Part 2: Dataset Access

Transaction data can be retrieved in two ways depending on where the process is executed.

**Option 1: Local environment**
- Uses a local file path
- Suitable when running the analysis locally

**Option 2: Azure Machine Learning workspace**
- Uses the Azure configuration file to connect to the workspace
- Retrieves the dataset registered in the Azure ML environment

In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


---
### Step 3: Prepare the Data

Before analysis begins, the transaction data is organized and adjusted for use in the fraud detection process.
- Adjusts the **transaction amount** so values can be compared fairly
- Removes the **time** field, which is not required for this proof of concept
- Separates the known fraud outcome so accuracy can be evaluated later

**Why this step is necessary:**
Well-prepared data improves the quality of the analysis and reduces the risk of misleading results.

In [5]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

---
### Step 4: Train the Fraud Detection Process

Machine learning is applied to examine transaction patterns and detect unusual behavior.
- Uses the **Isolation Forest** algorithm to identify transactions that differ significantly from normal activity
- Flags transactions that appear abnormal compared with typical patterns
- Labels each transaction as **normal (0)** or **potentially fraudulent (1)**

**Why this approach is used:**
Isolation Forest is a commonly used method for **anomaly detection in financial data** because fraud cases are rare and often appear as unusual patterns compared with normal transactions.

#### Operational Risk Considerations

Machine learning fraud detection models may face several operational risks:
- Fraud patterns may evolve over time, reducing model effectiveness
- Highly imbalanced data can make fraud detection difficult
- Over-reliance on automated decisions could lead to incorrect actions

To mitigate these risks, organizations typically:
- Retrain models regularly using updated transaction data
- Monitor detection performance continuously
- Maintain human fraud analysts to review flagged transactions

In [6]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

---
### Step 5: Evaluate the Results
The detection results are reviewed to understand how effectively fraud is identified.
- Compares predicted fraud alerts with known fraud outcomes
- Summarizes results in a performance table

**What this reveals:**
It shows how reliable the detection process is before it could be used in real financial monitoring.

#### Results Summary
The table below summarizes how the fraud detection process performed on the test data.
| Class | Description           | Precision | Recall | F1-Score | Support |
|-------|------------------------|-----------|--------|----------|---------|
| `0`   | Normal transactions    | **1.00**  | **1.00** | **1.00**   | 284,315 |
| `1`   | Fraudulent transactions| **0.29**  | **0.28** | **0.28**   | 492     |

**Understanding the Metrics**<br>
**Precision** measures how often flagged transactions are truly fraud, while **recall** measures how many actual fraud cases the system successfully detects. The **F1-score** provides a combined view of detection performance.

#### Interpreting the Results
Overall, this proof of concept is strong at recognizing normal activity, but not yet effective enough at catching fraud.
- The process is very strong at recognizing **normal transactions** and makes very few mistakes in that area.
  - Precision, Recall, and F1-score values of **1.00 for normal transactions**, meaning the model almost never misclassified legitimate activity in this test.
- However, it is much less effective at identifying **fraudulent transactions**.
  - It is correct only **29%** of the time (**Precision = 0.29**).
  - It identifies only **28%** of actual fraud cases (**Recall = 0.28**).

#### Understanding Overall Accuracy
The process appears very accurate overall, at about **99.9%**. However, this number can be misleading in fraud detection because most transactions are legitimate.
- The **Support** column shows there are **far more normal transactions than fraudulent ones**.
- Because of this imbalance, the process can appear highly accurate simply by correctly identifying normal transactions.

To understand the real performance, the **Fraudulent transactions** row should be reviewed. The metrics in that row provide a clearer view of how effectively the process identifies fraud.

#### Business Impact: False Positives vs Missed Fraud

In fraud detection, the system must balance two types of errors:

- **False positives** – legitimate transactions incorrectly flagged as fraud.<br>This can interrupt customer purchases and increase customer support workload.
- **Missed fraud** – fraudulent transactions that the system fails to detect.<br>These can lead to direct financial loss and reputational damage.

In practice, financial institutions generally prefer to review more suspicious transactions rather than risk missing fraudulent activity. As a result, fraud detection systems are often configured to **prioritize identifying potential fraud**, even if this leads to a higher number of alerts that require investigation.

In [7]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.29      0.28      0.28       492

    accuracy                           1.00    284807
   macro avg       0.64      0.64      0.64    284807
weighted avg       1.00      1.00      1.00    284807



---
### Step 6: Save the Fraud Detection Process (Optional)
The trained fraud detection process can be stored for future use.
- Registers the process in the **Azure Machine Learning workspace**
- Makes it available for reuse or further testing

**Practical benefit:**
The process can be reused later without rebuilding it.

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


---
### Step 7: Visualize and Interpret Detection Results

Finally, visual charts are used to better understand how the detection process behaves.
- Shows how many transactions were flagged as suspicious
- Compares transaction amounts between normal and flagged transactions
- Highlights factors that influence fraud alerts

**What this helps illustrate:**
These charts provide a clearer picture of the patterns driving fraud alerts and where the detection process may need improvement.

#### Chart 1: Count of Predicted Fraud Cases

**Purpose**: Quick check of how aggressive the system is at flagging anomalies.

![Count of Predicted Anomalies](img/chart_countplot.png)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


#### Chart 2: Transaction Amount by Prediction

**Purpose**: Shows whether unusual transaction amounts influence fraud alerts.

![Transaction Amount by Prediction Class](img/chart_boxplot.png)

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


#### Chart 3: Factors Influencing Fraud Alerts

**Purpose**: Helps explain what patterns the system is reacting to.

![Impact on model prediction](img/chart_shap_beeswarm.png)

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

---
### Opportunities for Improving the Detection Model

This proof of concept demonstrates the workflow for anomaly-based fraud detection, but additional improvements could increase performance:
- Incorporating more transaction characteristics such as merchant category or geographic patterns
- Testing additional algorithms such as Random Forest or Gradient Boosting
- Tuning model parameters to improve fraud detection sensitivity
- Combining multiple models to improve reliability

These improvements could help increase the system's ability to detect fraud while controlling false alerts.

### Model Limitations

This proof of concept demonstrates how machine learning can assist in identifying suspicious transactions, but it is not intended to make final fraud decisions on its own.

The system highlights **transactions that appear unusual based on historical patterns**. While this can significantly improve fraud monitoring, it is important to recognize that:
- Some fraudulent transactions may still go undetected.
- Some legitimate transactions may be flagged for review.
- Fraud alerts generated by the system should be **reviewed by fraud analysts before action is taken**.

For this reason, machine learning should be viewed as a **tool that supports fraud investigation teams**, helping them focus attention on higher-risk transactions rather than replacing human judgment.